# Intro to RAG pipelines

In [5]:
!uv add langchain-ollama

Resolved 75 packages in 1ms
Checked 68 packages in 0.59ms


In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="ornith-1.5:9b", base_url="http://localhost:11434", temperature=0.1)

In [3]:
from IPython.display import display, Markdown
result = llm.invoke("Quem é Sherlock Holmes?")
display(Markdown(result.content))


Sherlock Holmes é um **detetive fictício** criado pelo escritor britânico **Sir Arthur Conan Doyle**. Ele apareceu pela primeira vez em 1887, no conto "Um Estudo em Escarlate".

## Características principais

**Personalidade e método:**
- Extremamente observador e racional
- Famoso pelo método de **dedução** — chega a conclusões baseadas em pequenos detalhes que outros não percebem
- Gosta de ciência, química e medicina
- Tem uma mente brilhante, mas também defeitos: é solitário, às vezes arrogante e tem uma vida pessoal complicada

**Aparência:**
- Cabelos cacheados e desalinhados
- Rosto marcado por cicatrizes de uma queda de cavalo
- Costumava usar um chapéu-de-abas-largas e um cachecol

## Personagens associados

- **Dr. John Watson** — seu companheiro e narrador das histórias, um médico e melhor amigo
- **Professor Moriarty** — seu principal vilão, descrito como "o Napoleão do crime"
- **Irmã Mary Morstan** — personagem importante em alguns casos

## Obras onde ele aparece

Sherlock Holmes protagonizou 4 romances e 56 contos de Doyle, incluindo:
- *O Caso do Círculo Vermelho*
- *O Bumerangue*
- *Os Quatro Sinos*
- *O Vale do Medo*

## Legado

Holmes se tornou um dos personagens mais populares da literatura de detetives e influenciou inúmeros outros detetives famosos, como o **Sherlock Holmes de Basil Rathbone** (filmes dos anos 1930-40) e o **Sherlock Holmes de Benedict Cumberbatch** (série *Sherlock*, 2010).

Você gostaria de saber mais sobre alguma história específica ou sobre algum aspecto particular do personagem?

In [4]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    base_url="http://localhost:11434", model="nomic-embed-text-v2-moe:latest"
)

In [8]:
result = embedding_model.embed_query("Quem é Sherlock Holmes?")
print(result)

[0.03046434, 0.0189809, -0.04618363, -0.00037145673, 0.029326903, -0.008694615, -0.050802574, 0.029355628, -0.015610213, 0.020645048, -0.0291308, -0.036489356, 0.05357741, 0.00084378786, 0.023622045, -0.08064386, 0.029090453, -0.0020195434, 0.025740772, 0.05036334, 0.057661526, 0.032921046, 0.03993349, -0.024216112, 0.002571287, 0.023780884, 0.0261563, -0.0128097795, 0.047095943, 0.032904033, -0.010681981, 0.036909796, -0.00051663053, 0.01190871, -0.0020830594, -0.00368897, -0.03932114, 0.026010728, 0.0097104795, 0.01518954, 0.010836549, 0.005902954, 0.015183526, 0.013944543, 0.017247925, 0.0035832387, -0.06550993, 0.03120733, 0.019850545, -0.012035124, 0.01994591, -0.0017588339, -0.031373497, -0.04605388, 0.038141448, -0.057281543, 0.05537421, -0.027998881, 0.02908977, 0.031673066, -0.01655057, 0.033299938, 0.09414602, -0.024796227, -0.0118426075, -0.03246028, -0.006041249, 0.018035829, -0.07435112, -0.02374129, -0.018628733, -0.01633585, -0.027638473, 0.045344085, -0.017316507, 0.025

# 1. Carregar dados e documentos

In [5]:
import json

with open("booklist.json", "r") as f:
    booklist = json.load(f)

In [6]:
!uv add langchain-community langchain-text-splitters

Resolved 146 packages in 7ms
Prepared 55 packages in 1.60s                                                jupyterlab                ------------------------------ 16.22 MiB/16.38 MiB    jupyterlab                ------------------------------ 8.80 MiB/16.38 MiB     jupyterlab                ------------------------------ 5.20 MiB/16.38 MiB     jupyterlab                ------------------------------ 2.06 MiB/16.38 MiB     jupyterlab                ------------------------------ 712.56 KiB/16.38 MiB   jupyterlab                ------------------------------ 382.78 KiB/16.38 MiB   jupyterlab                ------------------------------ 366.78 KiB/16.38 MiB   jupyterlab                ------------------------------ 302.78 KiB/16.38 MiB   mjupyterlab                ------------------------------ 254.78 KiB/16.38 MiB   jupyterlab                ------------------------------ 238.78 KiB/16.38 MiB   upyterlab                ------------------------------ 190.78 KiB/16.38 MiB   upyterlab     

In [7]:
from langchain_community.document_loaders import TextLoader
import os

documents = []
for book in booklist:
    loader = TextLoader(book["path"], encoding="utf-8")
    doc = loader.load()[0]
    doc.metadata["title"] = book["title"]
    doc.metadata["author"] = book["author"]
    doc.metadata["year"] = book["year"]
    doc.metadata["genre"] = book["genre"]
    doc.metadata["language"] = book["language"]
    documents.extend([doc])

print(f"Loaded {len(documents)} documents from {len(booklist)} books.")

Loaded 2 documents from 2 books.


/tmp/ipykernel_2010351/1526419405.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
print(documents[0].metadata)  # Print the metadata of the first document

{'source': 'books/alices_adventure_in_wonderland.txt', 'title': "Alice's Adventures in Wonderland", 'author': 'Lewis Carroll', 'year': 1865, 'genre': 'Fantasy', 'language': 'English'}


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=250)
chunks = splitter.split_documents(documents)

In [9]:
print("Total number of chunks created:", len(chunks))

Total number of chunks created: 675


In [6]:
print(chunks[500])  # Print the metadata of the first chunk

page_content='“Forgive me, Doctor; I forgot myself. You do not need any help. I am so
worried in my mind that I am apt to be irritable. If you only knew the
problem I have to face, and that I am working out, you would pity, and
tolerate, and pardon me. Pray do not put me in a strait-waistcoat. I
want to think and I cannot think freely when my body is confined. I am
sure you will understand!” He had evidently self-control; so when the
attendants came I told them not to mind, and they withdrew. Renfield
watched them go; when the door was closed he said, with considerable
dignity and sweetness:--

“Dr. Seward, you have been very considerate towards me. Believe me that
I am very, very grateful to you!” I thought it well to leave him in this
mood, and so I came away. There is certainly something to ponder over in
this man’s state. Several points seem to make what the American
interviewer calls “a story,” if one could only get them in proper order.
Here they are:--

Will not mention “drinkin

In [10]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embedding_model)
vector_store.add_documents(chunks)

['a9e6c6c5-037f-45ee-970e-6f7c282aaa06',
 '4d4acc70-4f84-49fa-9f5e-8472a6d5f2ea',
 '4baadf0b-0fd9-43d5-921a-7e99d5f75cd3',
 '183dcde4-0325-43d8-8ed8-fd3deb7fa233',
 '86d72297-fb5a-494e-9520-733f64f01c85',
 '0572d59f-da71-4210-b765-95ca5c446387',
 '7fb458e2-d06a-4b64-b7ef-c5f39218543c',
 'd5667642-adb7-417c-9a4e-25c8ce429a08',
 'de0c113d-b823-4c83-a8ca-89f60470af2d',
 '248f57e2-a527-4621-afe2-27a7aa1543b3',
 '0c8f6aae-c37c-4fa1-86c5-8475863b7aed',
 '2f5d8572-d3c0-4ad2-b08b-c2ecdceb4191',
 'fe22621f-817c-460d-96a6-5391fac6dc31',
 '43f9371f-b1cd-428b-82bb-b6f958f49a4d',
 '749d139c-9bd2-4a60-b806-f8d7a51e17f5',
 '3f95b778-d395-4272-8dd2-9003eeddaf60',
 '998e2fb3-5277-460e-a6cf-dc0e6861d85d',
 'e582ecaf-74ee-4c4f-9864-91f7f07561f1',
 'a486e93e-dd49-4c99-b9be-a7c8cd1a9191',
 'e43e85ed-920d-4bf6-a63d-3323facf14d4',
 '3f992579-292e-47ef-947b-20f5a4f9c4c5',
 '54b9d8e3-36ea-4ef2-beeb-568290f44103',
 '39798923-5368-43d9-80b5-735124e6a183',
 '84367bd3-ddda-4683-958a-0e9b78ccafda',
 '43744fd6-4f51-

In [12]:
vector_store.similarity_search("What is Alice doing when she first notices the White Rabbit?", k=6)

[Document(id='ee633288-7c30-459e-87fa-62d0db45973f', metadata={'source': 'books/alices_adventure_in_wonderland.txt', 'title': "Alice's Adventures in Wonderland", 'author': 'Lewis Carroll', 'year': 1865, 'genre': 'Fantasy', 'language': 'English'}, page_content='Oh dear, what nonsense I’m talking!”\n\nJust then her head struck against the roof of the hall: in fact she was\nnow more than nine feet high, and she at once took up the little golden\nkey and hurried off to the garden door.\n\nPoor Alice! It was as much as she could do, lying down on one side, to\nlook through into the garden with one eye; but to get through was more\nhopeless than ever: she sat down and began to cry again.\n\n“You ought to be ashamed of yourself,” said Alice, “a great girl like\nyou,” (she might well say this), “to go on crying in this way! Stop\nthis moment, I tell you!” But she went on all the same, shedding\ngallons of tears, until there was a large pool all round her, about\nfour inches deep and reaching h

In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k": 6}, search_type="similarity")
retriever.invoke("Who is Alice?")

[Document(id='35462f08-3fe5-4aa4-97c4-1ae1aa771653', metadata={'source': 'books/alices_adventure_in_wonderland.txt', 'title': "Alice's Adventures in Wonderland", 'author': 'Lewis Carroll', 'year': 1865, 'genre': 'Fantasy', 'language': 'English'}, page_content='First came ten soldiers carrying clubs; these were all shaped like the\nthree gardeners, oblong and flat, with their hands and feet at the\ncorners: next the ten courtiers; these were ornamented all over with\ndiamonds, and walked two and two, as the soldiers did. After these came\nthe royal children; there were ten of them, and the little dears came\njumping merrily along hand in hand, in couples: they were all\nornamented with hearts. Next came the guests, mostly Kings and Queens,\nand among them Alice recognised the White Rabbit: it was talking in a\nhurried nervous manner, smiling at everything that was said, and went\nby without noticing her. Then followed the Knave of Hearts, carrying\nthe King’s crown on a crimson velvet c

In [12]:
!uv add langchain-classic

Resolved 146 packages in 1ms
Checked 140 packages in 0.92ms


In [13]:
from langchain_classic.chains import RetrievalQA

chat = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True)

chat.invoke("Who is Alice?")

{'query': 'Who is Alice?',
 'result': 'Based on the context provided, Alice is the main character and narrator of this story (an excerpt from Lewis Carroll\'s *Alice\'s Adventures in Wonderland*).\n\nHere\'s what the text tells us about her:\n\n- **She\'s a curious, thoughtful young girl** who falls down a rabbit hole into a strange, fantastical world.\n\n- **She\'s experiencing confusion about her identity.** As she grows larger (having eaten something that made her "more than nine feet high"), she wonders aloud whether she\'s been changed. She muses: *"Who in the world am I? Ah, that\'s the great puzzle!"*\n\n- **She compares herself to other children** she knows—reasoning that she\'s not "Ada" (whose hair is in long ringlets) and not "Mabel" (who knows so little)—but can\'t quite figure out who she has become.\n\n- **She\'s polite and self-aware**, recognizing that the characters she meets are "only a pack of cards" and that she needn\'t be afraid of them.\n\n- **She\'s emotional an

In [14]:
result = chat.invoke("Who is the White Rabbit?")

In [15]:
from IPython.display import display, Markdown

print(result)

text = result['result']

display(Markdown(text))

{'query': 'Who is the White Rabbit?', 'result': 'Based on the context provided, the White Rabbit is a character in Wonderland who appears to be a **servant** figure. Here\'s what the text reveals about him:\n\n- **He serves the Duchess** — He addresses Alice as "Mary Ann," his housemaid, and is worried about keeping the Duchess waiting. He\'s carrying gloves and a fan, which are things a servant would fetch for his mistress.\n\n- **He\'s in attendance at the Queen\'s court** — He\'s dressed "splendidly" and is anxious about being late, even taking a **watch out of his waistcoat-pocket** to check the time. He\'s clearly a punctual, dutiful servant.\n\n- **He\'s connected to the Queen\'s court** — He mentions that the Duchess is "under sentence of execution" (for boxing the Queen\'s ears) and is rushing to the Queen\'s croquet game, where he\'s present among the court.\n\nSo the White Rabbit is essentially a **talking rabbit dressed in human clothing who functions as a servant/housekeepe

Based on the context provided, the White Rabbit is a character in Wonderland who appears to be a **servant** figure. Here's what the text reveals about him:

- **He serves the Duchess** — He addresses Alice as "Mary Ann," his housemaid, and is worried about keeping the Duchess waiting. He's carrying gloves and a fan, which are things a servant would fetch for his mistress.

- **He's in attendance at the Queen's court** — He's dressed "splendidly" and is anxious about being late, even taking a **watch out of his waistcoat-pocket** to check the time. He's clearly a punctual, dutiful servant.

- **He's connected to the Queen's court** — He mentions that the Duchess is "under sentence of execution" (for boxing the Queen's ears) and is rushing to the Queen's croquet game, where he's present among the court.

So the White Rabbit is essentially a **talking rabbit dressed in human clothing who functions as a servant/housekeeper** in Wonderland's court, attending to the Duchess and present at the Queen of Hearts' croquet game. He's a creature that behaves like a human servant, complete with a waistcoat, gloves, fan, and watch.